# Applied workflow: from a dataset to a checked model

A useful applied model begins with a decision, not with a training API. This notebook uses a small image-like dataset so the whole workflow stays inspectable: define the task, inspect the data, establish a baseline, train a PyTorch model, and examine the errors.

The goal is not to hide the mechanics behind a high-level trainer. The fast.ai `Learner` is valuable because it makes a reliable training loop easy to reuse; the paired PyTorch loop shows what that convenience is doing.

In [ ]:
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

seed = 7
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
sns.set_theme(style="whitegrid", context="notebook")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## A small but nontrivial classification problem

Each example is a noisy 16×16 image containing either a vertical or a horizontal bar. The label is easy to describe, but the pixels include nuisance variation: thickness, position, and noise change from example to example. This makes the dataset useful for demonstrating splits and errors without downloading a large corpus.

In [ ]:
def make_bars(n, size=16, noise=0.20):
    images = np.zeros((n, 1, size, size), dtype=np.float32)
    labels = np.random.randint(0, 2, size=n)
    for i, label in enumerate(labels):
        width = np.random.randint(2, 5)
        offset = np.random.randint(2, size - width - 2)
        if label == 0:
            images[i, 0, :, offset:offset + width] = 1.0
        else:
            images[i, 0, offset:offset + width, :] = 1.0
    images += np.random.normal(0, noise, images.shape).astype(np.float32)
    return np.clip(images, 0, 1), labels

X, y = make_bars(2400)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.20, stratify=y_train, random_state=seed)
print(X_train.shape, X_valid.shape, X_test.shape)
print(pd.Series(y_train).value_counts(normalize=True).rename('train proportion'))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.flat, X_train[:10], y_train[:10]):
    ax.imshow(image[0], cmap='gray', vmin=0, vmax=1); ax.set_title(f'label {label}'); ax.axis('off')
plt.tight_layout()

## Baseline before the neural network

A majority-class classifier is a necessary reference. With balanced labels it should score about 50% accuracy. A model is useful only if it improves on a baseline under the same split.

In [ ]:
majority = np.bincount(y_train).argmax()
baseline_accuracy = (y_test == majority).mean()
print(f"majority baseline accuracy: {baseline_accuracy:.3f}")

## The explicit PyTorch training loop

The loop has four visible responsibilities: put tensors on the device, compute logits, compute cross-entropy, and update parameters only during training. Validation is run without gradient tracking. This separation makes it possible to inspect where a high-level learner helps and where it could hide mistakes.

In [ ]:
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
valid_loader = DataLoader(TensorDataset(torch.tensor(X_valid), torch.tensor(y_valid)), batch_size=256)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=256)

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(),
    nn.Linear(16, 2),
).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

def evaluate(loader):
    model.eval(); total_loss = 0.0; correct = 0; count = 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device)); yb = yb.to(device)
            total_loss += loss_fn(logits, yb).item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item(); count += len(yb)
    return total_loss / count, correct / count

history = []
for epoch in range(12):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss_fn(model(xb), yb).backward()
        optimizer.step()
    train_loss, train_accuracy = evaluate(train_loader)
    valid_loss, valid_accuracy = evaluate(valid_loader)
    history.append((epoch + 1, train_loss, valid_loss, train_accuracy, valid_accuracy))

history = pd.DataFrame(history, columns=['epoch','train_loss','valid_loss','train_accuracy','valid_accuracy'])
print(history.tail(1).to_string(index=False))
sns.lineplot(data=history, x='epoch', y='train_loss', label='train loss')
sns.lineplot(data=history, x='epoch', y='valid_loss', label='validation loss')
plt.title('Training and validation loss'); plt.show()

## Error analysis

The final accuracy is only a summary. A confusion matrix shows which class is confused with which; the examples below show whether the errors are ambiguous, corrupted, or evidence of a data problem.

In [ ]:
model.eval(); predictions=[]; probabilities=[]
with torch.no_grad():
    for xb, _ in test_loader:
        logits = model(xb.to(device))
        probabilities.append(logits.softmax(1).cpu())
        predictions.append(logits.argmax(1).cpu())
predictions = torch.cat(predictions).numpy()
probabilities = torch.cat(probabilities).numpy()
print(classification_report(y_test, predictions, target_names=['vertical','horizontal']))
sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['vertical','horizontal'], yticklabels=['vertical','horizontal'])
plt.xlabel('predicted'); plt.ylabel('actual'); plt.title('Test confusion matrix'); plt.show()

wrong = np.flatnonzero(predictions != y_test)[:10]
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(X_test[i,0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f"true {y_test[i]} / pred {predictions[i]}\nP={probabilities[i].max():.2f}")
    ax.axis('off')
plt.tight_layout()

## Where fast.ai fits

The `Learner` is a structured wrapper around the same concerns: a model, dataloaders, loss, optimizer, metrics, callbacks, and a training loop. It is appropriate when those defaults make an experiment easier to repeat. The explicit loop remains the reference implementation when debugging a new objective or unusual training behaviour.

The optional cell below is intentionally guarded. The notebook remains runnable in the base repository environment; installing `fastai` enables the comparison without changing the underlying PyTorch model.

In [ ]:
try:
    from fastai.vision.all import Learner, DataLoaders, accuracy
    fastai_available = True
except ImportError:
    fastai_available = False
    print('fastai is not installed; the PyTorch workflow above is the complete runnable path.')
fastai_available